In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import json

def load_closing_prices(in_filename="high_close.json"):
    """Load the high prices from a JSON file."""
    with open(in_filename, 'r') as price_file:
        return json.load(price_file)

_percents = np.array([1.0, 0.1, 0.2, 0.3, 0.4]) # Max to play with in the market
_available = 7500 * _percents
_bear_watch = 1 # Tracks which percentage I'm on for watching dips
_bull_watch = 1 # Tracks which percentage up I'm watching for.
_account = [{"limit": 7500, "contribute": 0, "shares": []},
            {"limit": 7500, "contribute": 0, "shares": []},
            {"limit": 1e9, "contribute": 0, "shares": []}]

_etfs = ["UPRO", "VT", "DFAW", "AVGE"]
_check_etf_bear = "UPRO"
_check_etf_bull = "UPRO"
_high_closes = load_closing_prices()
print(_high_closes)

def get_close_stock_price(ticker_list):
    """Given a list of ticker tags, fetches closing price and returns as dictionary."""
    closing_prices = {}
    try:
        data = yf.download(
            ticker_list,
            period="10d",
            interval="1d",
            auto_adjust=False,
            progress=False,
            threads=True,
            rounding=True
        )
        if data is None or data.empty:
            print("No market data returned.")
            return closing_prices

        closes = data["Close"]
        for ticker in ticker_list:
            try:
                prices = closes[ticker]
                prices = prices.dropna()
                if not prices.empty:
                    closing_prices[ticker] = float(prices.iloc[-1])
                else:
                    print(f"No valid closing price for {ticker}")

            except (KeyError, TypeError):
                print(f"Unable to retrieve {ticker}")

    except Exception as e:
        print(f"Error downloading market data: {e}")
    return closing_prices

def save_closing_prices(high_prices, out_filename="high_close.json"):
    """Saves the high prices to a JSON file."""
    with open(out_filename,'w') as price_file:
        json.dump(high_prices, price_file, indent=2)
    print(f"Closing prices saved to {price_file}")

def purchase_dip_stock(cur_price):
    """Buys stock keeping a record of it."""
    global _bear_watch
    global _account
    global _available
    price = cur_price[_check_etf_bear]
    put_amount = 0.0
    while price <= (1 - _percents[_bear_watch]) * _high_closes[_check_etf_bear]:
        put_amount += _available[_bear_watch]
        _bear_watch += 1
        if _bear_watch > 4:
            print("{} has gone below the 40% threshold! Down {} percent.".format(_check_etf_bear, (1.0 - price/_high_closes[_check_etf]) * 100.0))
            break
    for ledger in _account[0:2]:
        if put_amount <= 0.0:
            print("Finished purchasing {} stock.".format(_check_etf_bear))
            break
        if ledger["limit"] > ledger["contribute"]:
            actual_put = min(put_amount, ledger["limit"] - ledger["contribute"])
            ledger["shares"].append({"stock": _check_etf_bear, "buy_price": price,
                                     "amount": actual_put/price})
            put_amount -= actual_put
            ledger["contribute"] += actual_put
            _available[0] -= actual_put
            print("Invested {}. Remaining available {}.".format(actual_put, _available[0]))
    return put_amount
    if put_amount > 0.0: # I'm thinking I should move this out to some other subroutine, I should also make a bear and bull check ETF variable.
        _account[2]["shares"].append({"stock": "VT",
                                      "buy_price": cur_price["VT"],
                                      "amount": put_amount/cur_price["VT"]})
        _account[2]["contribute"] += put_amount
        _available[0] -= put_amount
        print("Invested {}. Remaining available {}.".format(put_amount, _available[0]))


#latest_closes = get_close_stock_price(etfs)
latest_closes = load_closing_prices()

latest_closes[_check_etf_bear] *= 0.65

if (_bear_watch <= 4) and (latest_closes[_check_etf_bear] <= (1.0 - _percents[_bear_watch]) * _high_closes[_check_etf_bear]):
    print("{0} is below the {1} watch mark. Buying {0} stock at {2}.".format(
        _check_etf_bear, _percents[_bear_watch], latest_closes[_check_etf_bear]))
    remaining = purchase_dip_stock(latest_closes)
    if remaining > 0.0:
        _check_etf_bear = "VT"
elif (_bull_watch <= 4) and (latest_closes[_check_etf_bull]) >= (1.0 + _percents[_bull_watch]) * _high_closes[_check_etf_bull]:
    print("{0} is above the {1} watch mark. Selling {0} stock at {2}.".format(
        _check_etf_bull, _percents[_bull_watch], latest_closes[_check_etf_bull]))
    sell_up_stock(latest_closes[_check_etf_bull], _check_etf_bull)



{'UPRO': 140.4600067138672, 'VT': 155.39999389648438, 'DFAW': 82.95999908447266, 'AVGE': 98.21710205078125}
UPRO is below the 0.1 watch mark. Buying UPRO stock at 91.29900436401367.
Invested 7500. Remaining available 12500.0.
Invested 4500.0. Remaining available 8000.0.


In [18]:
_account

[{'limit': 7500,
  'contribute': 7500,
  'shares': [{'stock': 'UPRO',
    'buy_price': 91.29900436401367,
    'amount': 82.14766472257602}]},
 {'limit': 7500,
  'contribute': np.float64(4500.0),
  'shares': [{'stock': 'UPRO',
    'buy_price': 91.29900436401367,
    'amount': np.float64(49.28859883354561)}]},
 {'limit': 1000000000.0, 'contribute': 0, 'shares': []}]

In [19]:
_bear_watch

4

In [20]:
_bull_watch

1

In [21]:
_available

array([8000., 2000., 4000., 6000., 8000.])